**Archived: supplemental analyses for a separate "life history" paper (not this pipeline's own outputs).**

These cells are copied verbatim from the original monolithic `06C_visualizationEnvAdapt.ipynb`
(cells 111-238) and depend on objects computed earlier in that original notebook (`metadata`,
`commonID`, `ePCs`, `tmpTrait3`/`KG3_class`, `spTre.rooted.filtered3`, etc. - now split across
`src/S05_envPC_analysis.R` and the trimmed `06C_visualizationEnvAdapt.ipynb`). Not intended to
run standalone without first re-deriving those objects; kept unparameterized for reference.

# Supplemental analysis: life history paper Figures

In [ ]:
spTre3 = read.tree("/workdir/sh2246/p_phyloGWAS/output/PoaceaeTree_angiosperm353_astral3_filtered_20250416.nwk")
spTre.rooted3 = root(spTre3,"ASM1935983v1")

In [ ]:
tempTrop = ifelse(tmpTrait3%in%c("Af","Am","Aw","As","BWh","BSh"),"tropical","temperate")
names(tempTrop) = names(tmpTrait3)

In [ ]:
write.table(data.frame(assemblyID = names(tmpTrait3),KG3 = tmpTrait3,temp.trop = tempTrop),
            "/workdir/sh2246/p_phyloGWAS/output/assemblyClimateZone.txt",quote = F,col.names =T, row.names = F,sep="\t")

In [ ]:
lifeHistoryNumeric = as.numeric(metadata[commonID,]$lifeHistory=="annual")

In [ ]:
which(tapply(metadata$lifeHistory,metadata$latest_name,function(x) length(table(x,useNA = "ifany"))>1))

In [ ]:
lifeHistoryID = commonID[!is.na(lifeHistoryNumeric)]

In [ ]:
names(lifeHistoryNumeric) = commonID

In [ ]:
spTre.rooted.filtered3= keep.tip(spTre.rooted3,lifeHistoryID)

In [ ]:
# fit <- phytools::fastAnc(keep.tip(spTre.rooted.filtered,lifeHistoryID),na.omit(lifeHistoryNumeric),
#                          vars=TRUE, CI=TRUE,)
# td <- data.frame(node = nodeid(keep.tip(spTre.rooted.filtered,lifeHistoryID), lifeHistoryID),
#                life_history = na.omit(lifeHistoryNumeric))
# nd <- data.frame(node = names(fit$ace), life_history = fit$ace)
# d <- rbind(td, nd)
# d$node <- as.numeric(d$node)
# plotTree <- full_join(keep.tip(spTre.rooted.filtered,lifeHistoryID), d, by = 'node')

In [ ]:
# Subset tree to include only species with trait data
rooted_tree <- root(spTre.rooted.filtered3, outgroup = "ASM1935983v1", resolve.root = TRUE)
# rooted_tree$edge.length[rooted_tree$edge.length==0.01] = 0.1 
# Step 2: Make the tree fully dichotomous (resolve polytomies randomly)
dichotomous_tree <- multi2di(rooted_tree)

# Step 3: Subset to match trait data
tree.sub <- keep.tip(dichotomous_tree, names(na.omit(lifeHistoryNumeric)))
# Extract trait data and ensure it's in the right order
trait <- na.omit(lifeHistoryNumeric)
trait <- trait[tree.sub$tip.label]  # match the order of the tips

# Use ace() for discrete traits (method = "ML")
fit <- ace(trait, tree.sub, type = "discrete",model = "ARD")

# Create data frame for tips
td <- data.frame(node = 1:Ntip(tree.sub),
                 life_history = trait)

# Create data frame for internal nodes
nd <- data.frame(node = (Ntip(tree.sub) + 1):(Ntip(tree.sub) + Nnode(tree.sub)),
                 life_history = fit$lik.anc[,2])

# Combine tip and node data
d <- rbind(td, nd)
d$node <- as.numeric(d$node)
plotTree <- full_join(tree.sub, d, by = 'node')

In [ ]:
edgeState = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,2])

In [ ]:
options(repr.plot.width=10, repr.plot.height=8)
hist(d$life_history)

In [ ]:
options(repr.plot.width=10, repr.plot.height=8)
hist(as.numeric(edgeState[,1])-as.numeric(edgeState[,2]),breaks = seq(-1,1,0.01))

In [ ]:
A2P = as.numeric(edgeState[,1])>0.85&as.numeric(edgeState[,2])<0.15
P2A = as.numeric(edgeState[,1])<0.15&as.numeric(edgeState[,2])>0.85
A2P_soft = as.numeric(edgeState[,1])>0.5&as.numeric(edgeState[,2])<0.5
P2A_soft = as.numeric(edgeState[,1])<0.5&as.numeric(edgeState[,2])>0.5
sum(A2P)
sum(P2A)
sum(A2P_soft)
sum(P2A_soft)

In [ ]:
rhizomeDat = data.table::fread("/workdir/sh2246/p_phyloGWAS/data/grassBase_cleaned_rhizome.txt",header = T,data.table = F)

In [ ]:
rhizomeDatVec = rhizomeDat[,1102]
names(rhizomeDatVec) = rhizomeDat[,2]
rhizomeDatVec = rhizomeDatVec[lifeHistoryID]

In [ ]:
table(rhizomeDatVec)

In [ ]:
rhizomeDatVec = na.omit(rhizomeDatVec)

In [ ]:
LHbyRZ = table(lifeHistoryNumeric[lifeHistoryID],rhizomeDatVec[lifeHistoryID])

In [ ]:
colnames(LHbyRZ) = c("non-rhizomous","rhizomatous")
rownames(LHbyRZ) = c("Perennial","Annual")

In [ ]:
LHbyRZ

In [ ]:
which(lifeHistoryNumeric[lifeHistoryID]==1&rhizomeDatVec[lifeHistoryID]==1)

In [ ]:
pheatmap(LHbyRZ,cluster_rows = F,cluster_cols = F,fontsize =18,legend = F,
         display_numbers = T,number_format = '%0d',number_color = "black",
         filename = "/workdir/sh2246/p_phyloGWAS/output/figure/lifeHistory/Fig1c.png",
         width = 8.7*.7,height = 8.7*.7,units = "cm",pointsize = 6,res = 600)


In [ ]:
LHRZID = intersect(lifeHistoryID,names(rhizomeDatVec))

In [ ]:
LHS = paste(lifeHistoryNumeric[LHRZID],rhizomeDatVec[LHRZID], sep = "_")

In [ ]:
names(LHS) = LHRZID

In [ ]:
write.table(LHRZID[LHS%in%c("1_0","0_0")],
            "/workdir/sh2246/p_phyloGWAS/data/nonrhizomatous_assemblies_20260415.txt",
            quote = F,row.names = F,col.names = F)

In [ ]:
# # Subset tree to include only species with trait data
# rooted_tree <- root(spTre.rooted.filtered3, outgroup = "ASM1935983v1", resolve.root = TRUE)
# # rooted_tree$edge.length[rooted_tree$edge.length==0.01] = 0.1 
# # Step 2: Make the tree fully dichotomous (resolve polytomies randomly)
# dichotomous_tree <- multi2di(rooted_tree)

# Step 3: Subset to match trait data
tree.sub <- keep.tip(dichotomous_tree, LHRZID)
# Extract trait data and ensure it's in the right order
trait <- na.omit(LHS)
trait <- trait[tree.sub$tip.label]  # match the order of the tips

# Use ace() for discrete traits (method = "ML")
fit <- ace(trait, tree.sub, type = "discrete",model = "ARD")

# Create data frame for tips
td <- data.frame(node = 1:Ntip(tree.sub),
                 life_history = trait,prob = 1)

# Create data frame for internal nodes
nd <- data.frame(node = (Ntip(tree.sub) + 1):(Ntip(tree.sub) + Nnode(tree.sub)),
                 life_history = colnames(fit$lik.anc)[apply(fit$lik.anc,1,which.max)],
                 prob = apply(fit$lik.anc,1,function(x) x[which.max(x)]))

# Combine tip and node data
d <- rbind(td, nd)
d$node <- as.numeric(d$node)
plotTree <- full_join(tree.sub, d, by = 'node')

In [ ]:
edgeState = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,2])

In [ ]:
edgeProb = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,3])

In [ ]:
transitionCount = c()
for (i in seq(0.5,0.95,.05)){
    PN2PR = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
    PN2AN = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
    PN2AR = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

    AN2PR = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
    AN2PN = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
    AN2AR = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

    PR2PN = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
    PR2AN = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
    PR2AR = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

    transitionCount = rbind(transitionCount,sapply(list(PN2PR,PN2AN,PN2AR,AN2PR,AN2PN,AN2AR,PR2PN,PR2AN,PR2AR),sum))
}



In [ ]:
transitionCount

In [ ]:
i = 0.9
PN2PR = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
PN2AN = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
PN2AR = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

AN2PR = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
AN2PN = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
AN2AR = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

PR2PN = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
PR2AN = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
PR2AR = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

In [ ]:
i = 0.5
PN2PR_soft = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
PN2AN_soft = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
PN2AR_soft = edgeState[,1] == "0_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

AN2PR_soft = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_1" & edgeProb[,2] > i
AN2PN_soft = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
AN2AR_soft = edgeState[,1] == "1_0" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

PR2PN_soft = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "0_0" & edgeProb[,2] > i
PR2AN_soft = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_0" & edgeProb[,2] > i
PR2AR_soft = edgeState[,1] == "0_1" & edgeProb[,1] > i & edgeState[,2] == "1_1" & edgeProb[,2] > i

In [ ]:
table(d$life_history)

In [ ]:
compDupLabel = (metadata[lifeHistoryID,]$complete + metadata[lifeHistoryID,]$duplicated)/5565
names(compDupLabel) = lifeHistoryID

In [ ]:
transitionByTribe = c()
for (i in 1:nrow(highlightNodeDat2)){
    tmp1 = table(plotTree@phylo$edge[P2A,1]%in%phangorn::Descendants(plotTree@phylo,node = highlightNodeDat2[i,1],"all"))
    tmp2 = table(plotTree@phylo$edge[A2P,1]%in%phangorn::Descendants(plotTree@phylo,node = highlightNodeDat2[i,1],"all"))
    transitionByTribe = rbind(transitionByTribe,c(tmp1[2],tmp2[2]))
}

colnames(transitionByTribe) = c("P2A","A2P")
rownames(transitionByTribe) = highlightNodeDat2$Tribe

In [ ]:
transitionByTribe_soft = c()
for (i in 1:nrow(highlightNodeDat2)){
    tmp1 = table(plotTree@phylo$edge[P2A_soft,1]%in%phangorn::Descendants(plotTree@phylo,node = highlightNodeDat2[i,1],"all"))
    tmp2 = table(plotTree@phylo$edge[A2P_soft,1]%in%phangorn::Descendants(plotTree@phylo,node = highlightNodeDat2[i,1],"all"))
    transitionByTribe_soft = rbind(transitionByTribe_soft,c(tmp1[2],tmp2[2]))
}

colnames(transitionByTribe_soft) = c("P2A","A2P")
rownames(transitionByTribe_soft) = highlightNodeDat2$Tribe

In [ ]:
transitionByTribe_soft[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]
transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
bp = barplot(t(transitionByTribe_soft[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
             beside = T,horiz = T,col = scales::alpha(c("violet","cyan"),.08),xlim = c(0,48))
# text(t(transitionByTribe_soft[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
#       bp,t(transitionByTribe_soft[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
#      pos = 4)
bp2 = barplot(t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
             beside = T,horiz = T,col = c("violet","cyan"),legend.text = T,
             args.legend = list(x="bottomright",bty = 'n'),add = T)
text(t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
      bp2,t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
     pos = 4)

In [ ]:
png("/workdir/sh2246/p_phyloGWAS/output/figure/lifeHistory/Fig1b.png",width = 8.7,height = 8.7*1.3,units = "cm",
    pointsize = 6,res = 600)
par(mar = c(5,5,1,1))
bp = barplot(t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
             beside = T,horiz = T,col = c("violet","cyan"),legend.text = T,
             args.legend = list(x="bottomright",bty = 'n'),xlim = c(0,48))
text(t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
      bp,t(transitionByTribe[c("Oryzeae","Poeae","Triticeae","Cynodonteae","Paniceae","Andropogoneae"),]),
     pos = 4)
dev.off()

In [ ]:
plotTree

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

p <- ggtree(plotTree,layout = "rectangular", ladderize = T,size = .3,aes(color = life_history))+
        scale_color_continuous(low='forestgreen', high='orange') +
    geom_nodepoint(color=ifelse(679:1355%in%plotTree@phylo$edge[P2A,1],"violet",NA), alpha=1/2, size=2,shape = 18) +  
    geom_nodepoint(color=ifelse(679:1355%in%plotTree@phylo$edge[A2P,1],"cyan",NA), alpha=1/2, size=2,shape = 18) +
    geom_tippoint(color=ifelse(lifeHistoryNumeric[lifeHistoryID]==1,"orange","forestgreen"),
                       shape=19, size=.1)
p <- p + geom_cladelab(node=highlightNodeDat2$node, align=T,angle = 60,fontsize = 2,offset = .45,hjust = 0.3,vjust =2,
                       label=c("Andropogoneae","Cynodonteae","Oryzeae","Paniceae","Poeae","Triticeae"))
p1 <- p+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(as.factor(lifeHistoryNumeric[lifeHistoryID])) , offset=0, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_manual(values = c('forestgreen','orange'),name = "Life History")
p1 <- p1+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(as.factor(rhizomeDatVec)), offset=0.11, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_manual(values = c("grey",'brown'),name = "Rhizome",na.value = "white")
p1 <- p1+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(tempTrop[lifeHistoryID]) , offset=.22, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_manual(values = c("royalblue","salmon"),name = "Temperate/Tropical")
p1 <- p1+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(compDupLabel), offset=.33, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, .3))+
    xlim(0,1.25) +
    scale_fill_viridis_c(option = "plasma",direction = -1,name = "Assembly completeness") +
    theme(axis.text=element_text(size=5),legend.position = "left",legend.key.width = unit(3,units = "mm"),
          legend.key.height = unit(3,units = "mm"),
          legend.margin = margin(t = -2,r =0,b = 0,l =0, unit='mm'),
          plot.margin = margin(t= 1, r= 1,b = 1,l = 1, unit='mm'),
          legend.title = element_text(size = 8),legend.text = element_text(size = 6))
p1

png("/workdir/sh2246/p_phyloGWAS/output/figure/lifeHistory/Fig1a.png",width = 8.7*1.6,height = 17.4,units = "cm",
    pointsize = 6,res = 600)
p1
dev.off()


In [ ]:
table(ifelse(nd$node%in%plotTree@phylo$edge[PN2AN,1],"violet",NA))

In [ ]:
transitionNodes = c(plotTree@phylo$edge[PN2AN,1],
                    plotTree@phylo$edge[AN2PN,1],
                    plotTree@phylo$edge[PN2PR,1],
                    plotTree@phylo$edge[PR2PN,1],
                    plotTree@phylo$edge[PR2AN,1],
                    plotTree@phylo$edge[PR2AR,1])
transitionNodesDat = data.frame(node = transitionNodes,
                                type = c(rep("1",sum(PN2AN)),
                                         rep("2",sum(AN2PN)),
                                         rep("3",sum(PN2PR)),
                                         rep("4",sum(PR2PN)),
                                         rep("5",sum(PR2AN)),
                                         rep("6",sum(PR2AR))))

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

p <- ggtree(plotTree,layout = "rectangular", ladderize = T,size = .3,aes(color = life_history,alpha = prob))+
        scale_color_manual(values = c('forestgreen','purple','orange','brown'),name = "Life History")+
    geom_tippoint(shape=19, size=.1)+
    geom_hilight(data=transitionNodesDat, aes(node=node, fill=type),type = "roundrect")+
    scale_fill_manual(values = c("red","royalblue","purple4","tan4","cyan","hotpink"))
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[PN2AN,1],"red",NA), alpha=1, size=2,shape = 18) +  
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[AN2PN,1],"royalblue",NA), alpha=1, size=2,shape = 18) +
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[PN2PR,1],"purple4",NA), alpha=1, size=2,shape = 18) +
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[PR2PN,1],"tan4",NA), alpha=1, size=2,shape = 18) +
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[PR2AN,1],"cyan",NA), alpha=1, size=2,shape = 18) +
#     geom_nodepoint(color=ifelse(nd$node%in%plotTree@phylo$edge[PR2AR,1],"hotpink",NA), alpha=1, size=2,shape = 18) 
    
p1 <- p+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(as.factor(LHS)) , offset=0, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_manual(values = c('forestgreen','purple','orange','brown'),name = "Life History")+
    theme(axis.text=element_text(size=5),legend.position = "left",legend.key.width = unit(3,units = "mm"),
          legend.key.height = unit(3,units = "mm"),
          legend.margin = margin(t = -2,r =0,b = 0,l =0, unit='mm'),
          plot.margin = margin(t= 1, r= 1,b = 1,l = 1, unit='mm'),
          legend.title = element_text(size = 8),legend.text = element_text(size = 6))
p1

png("/workdir/sh2246/p_phyloGWAS/output/figure/lifeHistory/Fig1a_v2.png",width = 8.7*1.5,height = 17.4,units = "cm",
    pointsize = 6,res = 600)
p1
dev.off()


In [ ]:
plotTree_relabeled = plotTree@phylo

In [ ]:
merge_tab5=data.frame(assemblyID=plotTree_relabeled$tip.label)
merge_tab5=merge(merge_tab5,metadata,by = "assemblyID")
merge_tab5=merge_tab5[!duplicated(merge_tab5$assemblyID),]
rownames(merge_tab5)=merge_tab5$assemblyID
merge_tab5=merge_tab5[plotTree_relabeled$tip.label,]

plotTree_relabeled$tip.label[!is.na(merge_tab5$spTaxa)] = merge_tab5$spTaxa[!is.na(merge_tab5$spTaxa)]

In [ ]:
plotTree_relabeled = full_join(plotTree_relabeled, d, by = 'node')

In [ ]:
plotTree_relabeled@phylo$node.label[-1] = round(as.numeric(plotTree_relabeled@phylo$node.label[-1]),2)

In [ ]:
plotTree_relabeled@phylo$node.label = paste0(plotTree_relabeled@phylo$node.label,"; ",
                                             round(fit$lik.anc[,1],2),"-",round(fit$lik.anc[,2],2))

In [ ]:
class(plotTree_relabeled@phylo$node.label)

In [ ]:
options(repr.plot.width=10, repr.plot.height=20)
p <- ggtree(plotTree_relabeled,layout = "rectangular", ladderize = T,size = .3,aes(color = life_history))+
        scale_color_continuous(low='forestgreen', high='orange') +
    geom_nodepoint(color=ifelse(679:1355%in%plotTree@phylo$edge[P2A,1],"violet",NA), alpha=1/2, size=2,shape = 18) +  
    geom_nodepoint(color=ifelse(679:1355%in%plotTree@phylo$edge[A2P,1],"cyan",NA), alpha=1/2, size=2,shape = 18) +
    geom_tiplab(color=ifelse(lifeHistoryNumeric[lifeHistoryID]==1,"orange","forestgreen"),size = 1)+
    geom_nodelab(vjust=.5,hjust = 0.1, size=1,color = "red") +
    theme(legend.position = "none")

p

In [ ]:
png("/workdir/sh2246/p_phyloGWAS/output/angiosperm353_astral3_spLabeled_withLifeHistoryACE_20250416.png",
    height = 60,width = 20, unit = "cm",pointsize = 6,res = 600)
p
dev.off()

In [ ]:
highlightNodeDat2

In [ ]:
rownames(coldAdaptCount)[rowSums(coldAdaptCount)>20]


In [ ]:
cor(as.data.frame(ePCs$environmental.features)$bio12_Annual_Precipitation_quan50,
    ePCs$synthetic.environmental.traits[,1])^2
cor(as.data.frame(ePCs$environmental.features)$bio12_Annual_Precipitation_quan50,
    ePCs$synthetic.environmental.traits[,2])^2
ppcor::pcor(cbind(ePCs$synthetic.environmental.traits[,1],
                  as.data.frame(ePCs$environmental.features)$bio12_Annual_Precipitation_quan50,
                  as.data.frame(ePCs$environmental.features)$bio01_Annual_Mean_Temperature_quan50))$estimate^2

In [ ]:
cor(eTraits_filtered$bio01_Annual_Mean_Temperature_quan50,eTraits_filtered$bio12_Annual_Precipitation_quan50)

In [ ]:
length(spTre.rooted.filtered$tip.label)

In [ ]:
eTraits_filtered[c("10wheat_assembly_jagger","Hvulgare_FT262_BPGv2",'Pp-Kellogg1297-DRAFT-PanAnd-1.0',
                   "Avena_sativa_cv_Sang_v1_assembly","mabamboo_1.0","Setaria_viridis_v2.0",
                  "P.virgatum_v5","IRGSP-1.0","Ag-CAM1351-DRAFT-PanAnd-1.0",
                   "Sn-CAM1369-DRAFT-PanAnd-1.0","Sorghum_bicolor_NCBIv3",
                  "Td-FL_9056069_6-REFERENCE-PanAnd-2.0a","Zm-B73-REFERENCE-NAM-5.0"),c(2,11)]

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

par(mfrow = c(1,2))
plotTraitOnTree(eTraits_filtered$bio01_Annual_Mean_Temperature_quan50,
                commonID,spTre.rooted.filtered,visible_tip_labels = F,
                legend.title = "Annual Mean Temp.",col = c("blue","lightblue","red"))
plotTraitOnTree(eTraits_filtered$bio12_Annual_Precipitation_quan50,
                commonID,spTre.rooted.filtered,visible_tip_labels = F,direction = 'leftwards',
                legend.title = "Annual Precipitation",col = c("brown","lightgreen","forestgreen"))


options(repr.plot.width=10, repr.plot.height=10)

par(mfrow = c(1,2))
plotTraitOnTree(ePCs$synthetic.environmental.traits[commonID,1],
                commonID,spTre.rooted.filtered,visible_tip_labels = F,
                legend.title = "envPC1",col = c("blue","lightblue","red"))
plotTraitOnTree(scale(ePCs$synthetic.environmental.traits[commonID,2],center = T,scale = T),
                commonID,spTre.rooted.filtered,visible_tip_labels = F,direction = 'leftwards',
                legend.title = "envPC2",col = c("forestgreen","lightgreen","brown"))


png("/workdir/sh2246/p_phyloGWAS/output/envAdapt_Bio1Quan50_PhyloTree.png",width = 20,height = 80,units = "cm",res = 600,pointsize = 6)
plotTraitOnTree(eTraits_filtered$bio01_Annual_Mean_Temperature_quan50,
                commonID,spTre.rooted.filtered,visible_tip_labels = T,
                legend.title = "Annual Mean Temp.",col = c("blue","lightblue","red"),fontsize = .5)
dev.off()

png("/workdir/sh2246/p_phyloGWAS/output/envAdapt_Bio12Quan50_PhyloTree.png",width = 20,height = 80,units = "cm",res = 600,pointsize = 6)
plotTraitOnTree(eTraits_filtered$bio12_Annual_Precipitation_quan50,
                commonID,spTre.rooted.filtered,visible_tip_labels = T,
                legend.title = "Annual Mean Temp.",col = c("brown","lightgreen","forestgreen"),fontsize = .5)
dev.off()

In [ ]:
eTraits_filtered[which.min(eTraits_filtered[,11]),c(2,11)]

In [ ]:
rownames(eTraits_filtered)[grep('Pp',rownames(eTraits_filtered))]

# annualism & env.

In [ ]:
eTraits_filtered$assemblyID = rownames(eTraits_filtered)


In [ ]:
mergedData = merge(metadata,eTraits_filtered,by ="assemblyID")

In [ ]:
dim(eTraits_filtered)

In [ ]:
colnames(eTraits_filtered)[grep("Seasonality",colnames(eTraits_filtered))]

In [ ]:
mergedData = mergedData %>% filter(lifeHistory=='annual'|lifeHistory=='perennial')

In [ ]:
mergedData$lifeHistoryNumeric = as.numeric(as.factor(mergedData$lifeHistory))

In [ ]:
anova(lm(mergedData$lifeHistoryNumeric~mergedData$bio15_Precipitation_Seasonality_quan50))
anova(lm(mergedData$lifeHistoryNumeric~mergedData$bio04_Temperature_Seasonality_quan50))
anova(lm(mergedData$lifeHistoryNumeric~mergedData$bio04_Temperature_Seasonality_quan50+mergedData$bio15_Precipitation_Seasonality_quan50))
anova(lm(mergedData$lifeHistoryNumeric~mergedData$bio04_Temperature_Seasonality_quan50*mergedData$bio15_Precipitation_Seasonality_quan50*mergedData$bio01_Annual_Mean_Temperature_quan50))

In [ ]:
layout(matrix(1:4,2,2),widths = c(3,1),heights = c(1,3))
par(mar = c(1,5,1,1))
plot(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"]),
     type ='l',col = "red",main = '',xaxt ='n',yaxt = 'n',xlab = '',ylab = '',ylim = c(0,.02))
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"]),
     type ='l',col = "black")
par(mar = c(5,5,1,1))
plot(mergedData$bio15_Precipitation_Seasonality_quan50,
     mergedData$bio04_Temperature_Seasonality_quan50,
     pch  = 19, col = ifelse(mergedData$lifeHistoryNumeric=="1","red","black"),
     xlab = "Precipitation Seasonality",ylab = "Temperature Seasonality")
plot(1,1,type = 'n',axes = F,xlab = '',ylab ='')
par(mar = c(1,1,1,1))
legend(0.3,1.1,legend = c("Annual","Perennial"),col = c("red","black"),pch = 19,bty ='n',cex = 2)
par(mar = c(5,1,1,1))
d1 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"])
d2 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"])
plot(d1$y,d1$x,type ='l',col = "red",xlim = c(0,.0025),xlab = '',ylab = '',xaxt = 'n',yaxt = 'n')
points(d2$y,d2$x,type ='l',col = "black")

In [ ]:
mycol = rep(NA,nrow(mergedData))
mycol[mergedData$bio01_Annual_Mean_Temperature_quan50>18] = "darkred"
mycol[mergedData$bio01_Annual_Mean_Temperature_quan50<18] = "darkblue"
mypch = ifelse(mergedData$lifeHistoryNumeric=="1",1,19)

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

layout(matrix(1:4,2,2),widths = c(3,1),heights = c(1,3))
par(mar = c(1,5,1,1))
plot(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred",main = '',xaxt ='n',yaxt = 'n',xlab = '',ylab = '',ylim = c(0,.02),xlim = c(0,180),lty=2)
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue",lty =2)
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred")
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue")


par(mar = c(5,5,1,1))
plot(mergedData$bio15_Precipitation_Seasonality_quan50,
     mergedData$bio04_Temperature_Seasonality_quan50,
     pch  = mypch, col = mycol,
     xlab = "Precipitation Seasonality",ylab = "Temperature Seasonality")
plot(1,1,type = 'n',axes = F,xlab = '',ylab ='')
par(mar = c(1,1,1,1))
legend(0.15,1.2,legend = c("Tropical Annual","Temperate Annual","Tropical Perennial","Temperate Perennial"),
       col = c("darkred","darkblue","darkred","darkblue"),pch = c(1,1,19,19),lty=c(2,2,1,1),bty ='n',cex = 1.2)
par(mar = c(5,1,1,1))
d1 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d2 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
d3 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d4 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
plot(d1$y,d1$x,type ='l',col = "darkred",xlim = c(0,.005),ylim = c(0,1400),xlab = '',ylab = '',xaxt = 'n',yaxt = 'n',lty =2)
points(d2$y,d2$x,type ='l',col = "darkblue",lty= 2)
points(d3$y,d3$x,type ='l',col = "darkred")
points(d4$y,d4$x,type ='l',col = "darkblue")

In [ ]:
png("/workdir/sh2246/p_phyloGWAS/output/lifeHistorySeasonalityAss.png",width = 17.4,height = 17.4,units = "cm",res = 600,pointsize = 8)
layout(matrix(1:4,2,2),widths = c(3,1),heights = c(1,3))
par(mar = c(1,5,1,1))
plot(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred",main = '',xaxt ='n',yaxt = 'n',xlab = '',ylab = '',ylim = c(0,.02),xlim = c(0,180),lty=2)
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue",lty =2)
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred")
points(density(mergedData$bio15_Precipitation_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue")


par(mar = c(5,5,1,1))
plot(mergedData$bio15_Precipitation_Seasonality_quan50,
     mergedData$bio04_Temperature_Seasonality_quan50,
     pch  = mypch, col = mycol,
     xlab = "Precipitation Seasonality",ylab = "Temperature Seasonality")
plot(1,1,type = 'n',axes = F,xlab = '',ylab ='')
par(mar = c(1,1,1,1))
legend(0.17,1.1,legend = c("Tropical Annual","Temperate Annual","Tropical Perennial","Temperate Perennial"),y.intersp = 2,
       col = c("darkred","darkblue","darkred","darkblue"),pch = c(1,1,19,19),lty=c(2,2,1,1),bty ='n',pt.cex = 1,cex = 1.25,seg.len = 3)
par(mar = c(5,1,1,1))
d1 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d2 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
d3 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d4 = density(mergedData$bio04_Temperature_Seasonality_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
plot(d1$y,d1$x,type ='l',col = "darkred",xlim = c(0,.005),ylim = c(0,1400),xlab = '',ylab = '',xaxt = 'n',yaxt = 'n',lty =2)
points(d2$y,d2$x,type ='l',col = "darkblue",lty= 2)
points(d3$y,d3$x,type ='l',col = "darkred")
points(d4$y,d4$x,type ='l',col = "darkblue")
dev.off()

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

layout(matrix(1:4,2,2),widths = c(3,1),heights = c(1,3))
par(mar = c(1,5,1,1))
plot(density(mergedData$bio18_Precipitation_Warmest_Quarter_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred",main = '',xaxt ='n',yaxt = 'n',xlab = '',ylab = '',ylim = c(0,.005),xlim = c(0,1000))
points(density(mergedData$bio18_Precipitation_Warmest_Quarter_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue")
points(density(mergedData$bio18_Precipitation_Warmest_Quarter_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "salmon")
points(density(mergedData$bio18_Precipitation_Warmest_Quarter_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "lightblue")


par(mar = c(5,5,1,1))
plot(mergedData$bio18_Precipitation_Warmest_Quarter_quan50,
     mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50,
     pch  = 19, col = mycol,xlim = c(0,1000),ylim = c(0,30),
     xlab = "Precipitation Warmest Quarter (12.9%)",ylab = "Temperature Wettest Quarter (5.3%)")
plot(1,1,type = 'n',axes = F,xlab = '',ylab ='')
par(mar = c(1,1,1,1))
legend(0.1,1.2,legend = c("Tropical Annual","Temperate Annual","Tropical Perennial","Temperate Perennial"),
       col = c("darkred","darkblue","salmon","lightblue"),pch = 19,bty ='n',cex = 1.5)
par(mar = c(5,1,1,1))
d1 = density(mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d2 = density(mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
d3 = density(mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d4 = density(mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
plot(d1$y,d1$x,type ='l',col = "darkred",xlim = c(0,.2),ylim = c(0,30),xlab = '',ylab = '',xaxt = 'n',yaxt = 'n')
points(d2$y,d2$x,type ='l',col = "darkblue")
points(d3$y,d3$x,type ='l',col = "salmon")
points(d4$y,d4$x,type ='l',col = "lightblue")

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

layout(matrix(1:4,2,2),widths = c(3,1),heights = c(1,3))
par(mar = c(1,5,1,1))
plot(density(mergedData$envPC_2[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "darkred",main = '',xaxt ='n',yaxt = 'n',xlab = '',ylab = '',ylim = c(0,.5),xlim = c(-3,3))
points(density(mergedData$envPC_2[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "darkblue")
points(density(mergedData$envPC_2[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18]),
     type ='l',col = "salmon")
points(density(mergedData$envPC_2[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18]),
     type ='l',col = "lightblue")


par(mar = c(5,5,1,1))
plot(mergedData$envPC_2,
     mergedData$envPC_8,
     pch  = 19, col = mycol,xlim = c(-3,3),ylim = c(-3,3),
     xlab = "envPC2 (8.9%)",ylab = "envPC8 (5.6%)")
plot(1,1,type = 'n',axes = F,xlab = '',ylab ='')
par(mar = c(1,1,1,1))
legend(0.1,1.2,legend = c("Tropical Annual","Temperate Annual","Tropical Perennial","Temperate Perennial"),
       col = c("darkred","darkblue","salmon","lightblue"),pch = 19,bty ='n',cex = 1.5)
par(mar = c(5,1,1,1))
d1 = density(mergedData$envPC_8[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d2 = density(mergedData$envPC_8[mergedData$lifeHistoryNumeric=="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
d3 = density(mergedData$envPC_8[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50>18])
d4 = density(mergedData$envPC_8[mergedData$lifeHistoryNumeric!="1"&mergedData$bio01_Annual_Mean_Temperature_quan50<18])
plot(d1$y,d1$x,type ='l',col = "darkred",xlim = c(0,.6),ylim = c(-3,3),xlab = '',ylab = '',xaxt = 'n',yaxt = 'n')
points(d2$y,d2$x,type ='l',col = "darkblue")
points(d3$y,d3$x,type ='l',col = "salmon")
points(d4$y,d4$x,type ='l',col = "lightblue")

In [ ]:
ppcor::pcor(cbind(mergedData$lifeHistoryNumeric,
                  mergedData$bio18_Precipitation_Warmest_Quarter_quan50,
                  mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50))$estimate^2

In [ ]:
ppcor::pcor(cbind(mergedData$lifeHistoryNumeric,
                  mergedData$envPC_2,
                  mergedData$envPC_8))$estimate^2

In [ ]:
mycol2 = rep(NA,nrow(mergedData))
mycol2[mergedData$latest_name%in%"Zea diploperennis"] = "yellow"
mycol2[mergedData$latest_name%in%"Zea mays"] = "darkorange"
mycol2[mergedData$latest_name%in%"Sorghum halepense"] = "salmon"
mycol2[mergedData$latest_name%in%"Sorghum bicolor"] = "darkred"
mycol2[mergedData$latest_name%in%"Oryza rufipogon"] = "lightgreen"
mycol2[mergedData$latest_name%in%"Oryza sativa"] = "darkgreen"
mycol2[mergedData$latest_name%in%"Panicum hallii"] = "lightblue"
mycol2[mergedData$latest_name%in%"Panicum miliaceum"] = "darkblue"
mycol2[mergedData$latest_name%in%"Thinopyrum elongatum"] = "pink"
mycol2[mergedData$latest_name%in%"Triticum aestivum"] = "purple"

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

plot(mergedData$bio18_Precipitation_Warmest_Quarter_quan50,
     mergedData$bio08_Mean_Temperature_Wettest_Quarter_quan50,
     pch  = 19, col = mycol2,xlim = c(0,1000),ylim = c(0,30),cex = 2,
     xlab = "Precipitation Warmest Quarter (12.9%)",ylab = "Temperature Wettest Quarter (5.3%)")
legend("bottomright",legend = c("Zea diploperennis","Zea mays",
                             "Sorghum halepense","Sorghum bicolor",
                             "Oryza rufipogon","Oryza sativa",
                             "Panicum hallii","Panicum miliaceum",
                             "Thinopyrum elongatum", "Triticum aestivum"),
       col = c("yellow","darkorange","salmon","darkred","lightgreen","darkgreen",
               "lightblue","darkblue","pink","purple"),
       pch = 19,cex = 1)

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

plot(mergedData$bio04_Temperature_Seasonality_quan50,
     mergedData$bio15_Precipitation_Seasonality_quan50,
     pch  = 19, col = mycol2,xlim = c(0,1000),ylim = c(0,150),cex = 2,
     xlab = "Temperature seasonality",ylab = "Precipitation seasonality")
legend("topright",legend = c("Zea diploperennis","Zea mays",
                             "Sorghum halepense","Sorghum bicolor",
                             "Oryza rufipogon","Oryza sativa",
                             "Panicum hallii","Panicum miliaceum",
                             "Thinopyrum elongatum", "Triticum aestivum"),
       col = c("yellow","darkorange","salmon","darkred","lightgreen","darkgreen",
               "lightblue","darkblue","pink","purple"),
       pch = 19,cex = 1)

In [ ]:
quan50Idx = grep("quan50",colnames(mergedData))

corRes = c()
for (i in quan50Idx){
    tmp = cor.test(mergedData$lifeHistoryNumeric,mergedData[,i])
    corRes = rbind(corRes,c(tmp$estimate,tmp$p.value))
}


In [ ]:
ePCIdx = grep("PC",colnames(mergedData))

corRes2 = c()
for (i in ePCIdx){
    tmp = cor.test(mergedData$lifeHistoryNumeric,mergedData[,i])
    corRes2 = rbind(corRes2,c(tmp$estimate,tmp$p.value))
}


In [ ]:
rownames(corRes) = gsub("_quan50","",colnames(mergedData)[quan50Idx])
rownames(corRes2) = colnames(mergedData)[ePCIdx]

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
par(mar = c(5,20,2,2))
barplot(tail(sort(-log10(corRes[,2])),20),horiz = T,las = 1)
barplot(tail(sort(corRes[,1]^2),20),horiz = T,las = 1,xlab = expression(r^2))

In [ ]:
tail(sort(corRes[,1]^2),10)
tail(sort(corRes2[,1]^2),10)

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
par(mar = c(5,5,2,2))
barplot(tail(sort(-log10(corRes2[,2])),20),horiz = T,las = 1)
barplot(tail(sort(corRes2[,1]^2),20),horiz = T,las = 1,xlab = expression(r^2))
png("/workdir/sh2246/p_phyloGWAS/output/envPC_perenniality_association.png",width = 16,height = 16, units = "cm",res = 600,pointsize = 8)
par(mar = c(5,8,2,2))
barplot(tail(sort(corRes2[,1]^2),20),horiz = T,las = 1,xlab = expression(r^2))
dev.off()

In [ ]:
write.table(ePCs$variable.components,"/workdir/sh2246/p_phyloGWAS/output/envPC_loading_mat.txt",sep = "\t",col.names = T,row.names = T,quote = F)

In [ ]:
tmpX = apply(eTraits_filtered[,1:285],2,function(x) scale(x))
tmpX[is.na(tmpX)] = 0

In [ ]:
y2 = ePCs$variable.components[seq(2,285,3),3]%*%t(tmpX[,seq(2,285,3)])

In [ ]:
plot(y2[1,]/sd(y2[1,]),eTraits_filtered$envPC_3)
cor(y2[1,],eTraits_filtered$envPC_3)

In [ ]:
ePCIdx

In [ ]:
summary(lm(as.numeric(traitMat[commonID2,2])~eTraits_filtered[commonID2,287]+eTraits_filtered[commonID2,293] +eTraits_filtered[commonID2,294]))

In [ ]:
tail(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,2]^2),]),20)

In [ ]:
tail(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,8]^2),]),20)

In [ ]:
tail(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,9]^2),]),10)

In [ ]:
head(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,1]),]),10)
tail(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,1]),]),10)

In [ ]:
head(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,2]),]),20)
tail(na.omit(ePCs$variable.correlation[order(ePCs$variable.correlation[,2]),]),20)

In [ ]:
ePCs$variable.correlation["bio01_Annual_Mean_Temperature_quan50",2]

In [ ]:
corMat = cor(eTraits_filtered[,!apply(eTraits_filtered,2, function(x) all(x==0))])
corMat2 = cor(eTraits_filtered[,!apply(eTraits_filtered,2, function(x) all(x==0))])^2

In [ ]:
options(repr.plot.width = 15,repr.plot.height = 15)
pheatmap(corMat)

In [ ]:
pheatmap(corMat,filename = "/workdir/sh2246/p_phyloGWAS/output/corMat_env.png",width = 30,height = 30,units = "cm",res = 600,border_color = NA,fontsize_row = 6,fontsize_col = 6)

In [ ]:
pheatmap(corMat2,filename = "/workdir/sh2246/p_phyloGWAS/output/corMat2_env.png",width = 30,height = 30,units = "cm",res = 600,border_color = NA,fontsize_row = 6,fontsize_col = 6)

In [ ]:
annot_labels = data.frame(categories = c(rep(env_metadata[,3],each = 3),rep("envPC",40)))
rownames(annot_labels) = c(paste(rep(env_metadata[,1],each = 3),c("quan10","quan50","quan90"),sep = "_"),paste0("envPC_",1:40))

In [ ]:
annot_colors = list(categories = c(combined = "grey", 
                                   envPC = "purple", 
                                   growth = "forestgreen",
                                   precipitation = "cyan",
                                   soil_feature = "peru",
                                   soil_temperature = "salmon",
                                   temperature = "brown"))

In [ ]:
pheatmap(corMat2,border_color = NA,fontsize_row = 4,fontsize_col = 4,annotation_row = annot_labels,
         annotation_col = annot_labels,annotation_colors = annot_colors,
         filename = "/workdir/sh2246/p_phyloGWAS/output/corMat2_env_labeled.png",width = 20,height = 20,units = "cm",res = 600)

In [ ]:
options(repr.plot.width = 8,repr.plot.height = 8)
plot(eTraits_filtered$bio01_Annual_Mean_Temperature_quan50,eTraits_filtered$bio12_Annual_Precipitation_quan50,
     xlab = "Annual temperature (C)",ylab = "Annual precipitation (mm)",pch = 19 ,col = )

In [ ]:
ggplot(eTraits_filtered, aes(x = bio01_Annual_Mean_Temperature_quan50, 
                             y = bio12_Annual_Precipitation_quan50)) +
  geom_point()

In [ ]:
library(gridExtra)
options(repr.plot.width = 18,repr.plot.height = 6)
p1 = ggplot(eTraits_filtered, aes(x = bio01_Annual_Mean_Temperature_quan50, 
                             y = bio12_Annual_Precipitation_quan50, colour = envPC_1)) +
    geom_point(size = 4)+
    xlab("Annual Mean Temperature (\u00B0C)")+
    ylab("Annual Precipitation (mm)")+
    scale_color_gradientn(colors = c("blue","lightblue","red"),name = "envPC1")+
    theme_bw()+
    theme(axis.title = element_text(size = 16),axis.text = element_text(size = 10))
p2 = ggplot(eTraits_filtered, aes(x = bio01_Annual_Mean_Temperature_quan50, 
                             y = bio12_Annual_Precipitation_quan50, colour = envPC_2)) +
    geom_point(size = 4)+
    xlab("Annual Mean Temperature (\u00B0C)")+
    ylab("Annual Precipitation (mm)")+
    scale_color_gradientn(colors = c("blue","lightblue","red"),name = "envPC2")+
    theme_bw()+
    theme(axis.title = element_text(size = 16),axis.text = element_text(size = 10))
grid.arrange(p1, p2, nrow = 1)

In [ ]:
ggplot(eTraits_filtered, aes(x = bio01_Annual_Mean_Temperature_quan50, 
                             y = bio12_Annual_Precipitation_quan50, colour = envPC_2)) +
    geom_point()+
    xlab("Annual Mean Temperature (\u00B0C)")+
    ylab("Annual Precipitation (mm)")+
    scale_color_gradientn(colors = c("blue","lightblue","red"),name = "envPC2")+
    theme_bw()

In [ ]:
ggplot(eTraits_filtered, aes(x = bio01_Annual_Mean_Temperature_quan50, 
                             y = bio12_Annual_Precipitation_quan50, colour = envPC_2)) +
  geom_point()

In [ ]:
ggplot(eTraits_filtered, aes(x = envPC_1, 
                             y = envPC_2, colour = bio01_Annual_Mean_Temperature_quan50)) +
  geom_point()

In [ ]:
ggplot(eTraits_filtered, aes(x = envPC_1, 
                             y = envPC_2, colour = bio12_Annual_Precipitation_quan50)) +
  geom_point()

In [ ]:
ggplot(eTraits_filtered, aes(x = envPC_1, 
                             y = envPC_2, colour = rid_quan50)) +
  geom_point()

In [ ]:
colnames(eTraits_filtered)

In [ ]:
tmp$estimate

In [ ]:
pheatmap(cor(eTraits_filtered[,paste0(names(tail(sort(corRes[,1]^2),20)),"_quan50")],use = "complete"),
         display_numbers = T,cluster_rows = T,cluster_cols = T)

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
par(mfrow = c(1,2))
boxplot(eTraits_filtered[commonID2,]$bio18_Precipitation_Warmest_Quarter_quan50~as.numeric(traitMat[commonID2,2]),
        names = c("Annual","Perennial"),xlab = "life history",ylab = "Precipitation of the warmest quarter")
boxplot(eTraits_filtered[commonID2,]$bio08_Mean_Temperature_Wettest_Quarter_quan50~as.numeric(traitMat[commonID2,2]),
        names = c("Annual","Perennial"),xlab = "life history",ylab = "Mean temperature of the wettest quarter")


In [ ]:
table(as.numeric(traitMat[,2]))